In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as Patch
from matplotlib.colors import to_rgb

# --------------------------------------------------
# 1. Load and reshape your data
# --------------------------------------------------
csv_path = "/Users/celsapardoaraujo/Documents/data/magnetar-br/magnetar_fraction_plausible_piechart.csv"  # change path if needed
df = pd.read_csv(csv_path)

In [ ]:
df.columns

In [ ]:
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
yellow = plt.cm.Set2(5 / 7)
orange = plt.cm.Set2(1 / 7)
sea = plt.cm.Set2(0 / 7)
blue = plt.cm.Set2(2 / 7)
green = plt.cm.Set2(4 / 7)
purple = plt.cm.Set2(3 / 7)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as Patch

# ============================================================
# 1. DATA
# ============================================================

# df should have columns:
#  "CC", "B_0", "Feasible magnetar fraction [%]"
# where the last one is like "0", "60", "[30-60]" or "[30–60]"

df = df.sort_values(by=["CC", "B_0"])

def parse_value(v):
    """
    '0'         -> None (no bar)
    '60'        -> (0, 60)
    '[30-60]'   -> (30, 60)
    '[30–60]'   -> (30, 60)  (hyphen or en-dash)
    """
    # numeric directly
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if v == 0:
            return None
        return (0.0, float(v))

    # strings
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s == "0":
            return None

        # bracketed interval
        if s.startswith("[") and s.endswith("]"):
            inner = s[1:-1]
            if "–" in inner:
                lo, hi = inner.split("–")
            elif "-" in inner:
                lo, hi = inner.split("-")
            else:
                return None
            return float(lo), float(hi)

        # plain number
        try:
            val = float(s)
            if val == 0:
                return None
            return (0.0, val)
        except ValueError:
            return None

    return None

# One row per (B0, CC)
df["range"] = df["Feasible magnetar fraction [%]"].map(parse_value)

field_order = list(dict.fromkeys(df["B_0"]))  # order of B0
ccsn_order  = list(dict.fromkeys(df["CC"]))   # order of CC

n_cc   = len(ccsn_order)   # e.g. 3
n_b0   = len(field_order)  # e.g. 4

# Total angular slices: one per (CC × B0)
total_slices = n_cc * n_b0
r_max = 100
slice_width = 2 * np.pi / total_slices

# ============================================================
# 2. FIGURE & BACKGROUND GRADIENT
# ============================================================
base_colors = [
    "#E76F51",   # reddish for CC=first
    "#457B9D",   # bluish for CC=second
    "#2A9D8F",   # greenish for CC=third
]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
ax.spines['polar'].set_visible(False)
ax.spines['polar'].set_linewidth(0)

ax.set_theta_direction(-1)   # clockwise
r_plot = 120
ax.set_ylim(0, r_plot)

ax.set_xticks([])

def make_shades(hex_color):
    """Return 4 shades (brightest → darkest) from a base hex color."""
    r, g, b = to_rgb(hex_color)
    darkest = (r * 0.45, g * 0.45, b * 0.45)
    dark    = (r * 0.65, g * 0.65, b * 0.65)
    medium  = (r * 0.80, g * 0.80, b * 0.80)
    light   = (r * 0.93, g * 0.93, b * 0.93)
    return [light, medium, dark, darkest]

slices_per_sector = n_b0       # per CC: 1 slice per B0
bands_per_sector  = n_b0       # 4 B0 values
slices_per_band   = 1          # 1 slice per band

# Background gradient: CC sector × B0 band
for sector_idx, base_col in enumerate(base_colors[:n_cc]):
    CC_SN_labels = ["CC SN $= 1$","CC SN $= 2$","CC SN $= 3$"]
    angles_text_cc = [np.pi/4, np.pi, np.pi+np.pi/1.5]
    rotation_text = [225,90,-30]
    
    ax.text(
            angles_text_cc[sector_idx],
            r_plot,
            CC_SN_labels[sector_idx],
            ha="center",
            va="center",
            color= make_shades(base_colors[sector_idx])[2],
            fontsize=20,
            rotation= rotation_text[sector_idx],
            rotation_mode="anchor",
            zorder=30,
        )
    sector_start = sector_idx * slices_per_sector
    shades = make_shades(base_col)

    for band_idx, b0 in enumerate(field_order):
        band_start_slice = sector_start + band_idx * slices_per_band
        band_end_slice   = band_start_slice + slices_per_band

        start_theta  = band_start_slice * slice_width
        width_theta  = (band_end_slice - band_start_slice) * slice_width
        center_theta = start_theta + width_theta / 2.0

        shade_color = shades[band_idx % len(shades)]

        ax.bar(
            center_theta,
            r_max,
            width=width_theta,
            bottom=0,
            color=shade_color,
            edgecolor="none",
            alpha=0.8,
            zorder=0,
        )

# Radial lines
for k in range(total_slices):
    theta = k * slice_width

    r_end = r_max
    lw = 0.8

    # CC boundaries every 'slices_per_sector' → longer + thicker
    if k % slices_per_sector == 0:
        r_end = r_plot
        lw = 2.0

    ax.plot(
        [theta, theta],
        [0, r_end],
        color="black",
        linewidth=lw,
        zorder=5
    )

# ============================================================
# 3. OVERLAY GREY BARS FROM SINGLE-CLASS DATA
# ============================================================

bar_color = "white"  # dark grey

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):

        slice_index = ic * slices_per_sector + ib * slices_per_band
        theta_center = (slice_index + 0.5) * slice_width

        row = df[
            (df["CC"] == ccsn)
            & (df["B_0"] == b0)
        ]

        if row.empty:
            continue

        r_range = row.iloc[0]["range"]

        # --- defensive checks to avoid polar Path errors ---
        if (
            r_range is None
            or not isinstance(r_range, (tuple, list))
            or len(r_range) != 2
        ):
            continue

        r_min, r_max_bar = r_range

        if not (np.isfinite(r_min) and np.isfinite(r_max_bar)):
            continue

        height = r_max_bar - r_min
        if height <= 0:
            continue

        ax.bar(
            theta_center,
            height,
            width=slice_width * 0.9,
            bottom=r_min,
            color=bar_color,
            edgecolor="0.1",
            linewidth=0.6,
            zorder=3,
        )

# ============================================================
# 4. LABEL INITIAL MAGNETIC FIELDS (one label per wedge)
# ============================================================

# Nice LaTeX labels for each B_0 in field_order
B_0_labels = [
    r"$10^{14}\,\mathrm{G}$",
    r"$1.5 \times 10^{14}\,\mathrm{G}$",
    r"$2 \times 10^{14}\,\mathrm{G}$",
    r"$2.5 \times 10^{14}\,\mathrm{G}$",
]
# Map actual B_0 numeric values to their label
b0_label_map = dict(zip(field_order, B_0_labels))

label_radius = r_max + 3

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):

        band_start   = ic * slices_per_sector + ib * slices_per_band
        band_end     = band_start + slices_per_band
        center_slice = (band_start + band_end) / 2.0

        theta_center = center_slice * slice_width

        angle_deg = np.degrees(np.pi/2 - theta_center) + 180

        label = b0_label_map.get(b0, f"{b0:g}")

        ax.text(
            theta_center,
            label_radius,
            label,
            ha="center",
            va="center",
            color="black",
            fontsize=17,
            rotation=angle_deg,
            rotation_mode="anchor",
            zorder=30,
        )

# ============================================================
# 5. CONCENTRIC CIRCLES WITH HIGHLIGHTS AT 10,30,50,70,90
# ============================================================
r_ticks = [11, 21, 31, 41, 51, 61, 71, 81, 91]
labels  = [10, 20, 30, 40, 50, 60, 70, 80, 90]
important_labels = {10, 30, 50, 70, 90}

ax.set_yticks(r_ticks)
ax.set_yticklabels([])
ax.grid(color="white", linestyle="--", linewidth=0.4, alpha=0.5)

theta = np.linspace(0, 2*np.pi, 400)

for r, lab in zip(r_ticks, labels):
    if lab in important_labels:
        ax.plot(
            theta,
            np.full_like(theta, r),
            color="white",
            linewidth=1.0,
            linestyle=":",
            alpha=0.9,
            zorder=5,
        )

theta_label = np.deg2rad(45)

for r, lab in zip(r_ticks, labels):
    ax.text(
        theta_label,
        r,
        str(lab) + '%',
        ha="center",
        va="center",
        color="white",
        fontsize=13 if lab in important_labels else 8,
        fontweight="bold" if lab in important_labels else "normal",
        zorder=10
    )

# ============================================================
# 6. LEGEND (single class)
# ============================================================
legend_handles = [
    Patch.Patch(
        facecolor=bar_color,
        edgecolor="black",
        label="Feasible magnetar fraction"
    ),
]

fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    frameon=False,
    fontsize=13
)

plt.tight_layout()
plt.savefig(f'/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/plot_piechart_magnetar_fraction.pdf')

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as Patch

# ============================================================
# 0. PATHS
# ============================================================
csv_path_with = "/Users/celsapardoaraujo/Documents/data/magnetar-br/magnetar_fraction_plausible_piechart.csv"
csv_path_wo   = "/Users/celsapardoaraujo/Documents/data/magnetar-br/magnetar_fraction_wihout_cco_plausible_piechart.csv"
out_path      = "/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/plot_piechart_magnetar_fraction_with_vs_without_cco.pdf"

VALUE_COL = "Feasible magnetar fraction [%]"  # change if your CSV uses another column name

# ============================================================
# 1. LOAD + PARSE
# ============================================================
df_with = pd.read_csv(csv_path_with)
df_wo   = pd.read_csv(csv_path_wo)

df_with = df_with.sort_values(by=["CC", "B_0"]).copy()
df_wo   = df_wo.sort_values(by=["CC", "B_0"]).copy()

def parse_value(v):
    """
    '0'         -> None
    '60'        -> (0, 60)
    '[30-60]'   -> (30, 60)
    '[30–60]'   -> (30, 60)  (hyphen or en-dash)
    """
    # numeric directly
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if float(v) == 0:
            return None
        return (0.0, float(v))

    # strings
    if isinstance(v, str):
        s = v.strip()
        if s == "" or s == "0":
            return None

        # bracketed interval
        if s.startswith("[") and s.endswith("]"):
            inner = s[1:-1].strip()
            if "–" in inner:
                lo, hi = inner.split("–")
            elif "-" in inner:
                lo, hi = inner.split("-")
            else:
                return None
            return float(lo), float(hi)

        # plain number
        try:
            val = float(s)
            if val == 0:
                return None
            return (0.0, val)
        except ValueError:
            return None

    return None

df_with["range"] = df_with[VALUE_COL].map(parse_value)
df_wo["range"]   = df_wo[VALUE_COL].map(parse_value)

# Use ordering from df_with (assumes both share same CC and B_0 grid)
field_order = list(dict.fromkeys(df_with["B_0"]))
ccsn_order  = list(dict.fromkeys(df_with["CC"]))

n_cc = len(ccsn_order)
n_b0 = len(field_order)

total_slices = n_cc * n_b0
slice_width  = 2 * np.pi / total_slices
r_max_data   = 100
r_plot       = 120

# ============================================================
# 2. FIGURE & BACKGROUND GRADIENT
# ============================================================
base_colors = [
    "#E76F51",   # reddish
    "#457B9D",   # bluish
    "#2A9D8F",   # greenish
]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)

ax.spines["polar"].set_visible(False)
ax.spines["polar"].set_linewidth(0)

ax.set_theta_direction(-1)  # clockwise
ax.set_ylim(0, r_plot)
ax.set_xticks([])

def make_shades(hex_color):
    r, g, b = to_rgb(hex_color)
    darkest = (r * 0.45, g * 0.45, b * 0.45)
    dark    = (r * 0.65, g * 0.65, b * 0.65)
    medium  = (r * 0.80, g * 0.80, b * 0.80)
    light   = (r * 0.93, g * 0.93, b * 0.93)
    return [light, medium, dark, darkest]

slices_per_sector = n_b0  # within each CC sector, one slice per B_0
slices_per_band   = 1     # each band is one slice

# CC labels (keep your style)
CC_SN_labels   = ["CC SN $= 1$", "CC SN $= 2$", "CC SN $= 3$"]
angles_text_cc = [np.pi/4, np.pi, np.pi + np.pi/1.5]
rotation_text  = [225, 90, -30]

for sector_idx, base_col in enumerate(base_colors[:n_cc]):
    ax.text(
        angles_text_cc[sector_idx],
        r_plot,
        CC_SN_labels[sector_idx] if sector_idx < len(CC_SN_labels) else f"CC SN $= {sector_idx+1}$",
        ha="center",
        va="center",
        color=make_shades(base_colors[sector_idx])[2],
        fontsize=20,
        rotation=rotation_text[sector_idx] if sector_idx < len(rotation_text) else 0,
        rotation_mode="anchor",
        zorder=30,
    )

    sector_start = sector_idx * slices_per_sector
    shades = make_shades(base_col)

    for band_idx, b0 in enumerate(field_order):
        band_start_slice = sector_start + band_idx * slices_per_band
        band_end_slice   = band_start_slice + slices_per_band

        start_theta  = band_start_slice * slice_width
        width_theta  = (band_end_slice - band_start_slice) * slice_width
        center_theta = start_theta + width_theta / 2.0

        shade_color = shades[band_idx % len(shades)]

        ax.bar(
            center_theta,
            r_max_data,
            width=width_theta,
            bottom=0,
            color=shade_color,
            edgecolor="none",
            alpha=0.8,
            zorder=0,
        )

# Radial lines
for k in range(total_slices):
    theta = k * slice_width
    r_end = r_max_data
    lw = 0.8
    if k % slices_per_sector == 0:  # CC boundaries
        r_end = r_plot
        lw = 2.0
    ax.plot([theta, theta], [0, r_end], color="black", linewidth=lw, zorder=5)

# ============================================================
# 3. RANGE BARS SIDE-BY-SIDE (WITH vs WITHOUT CCOs)
# ============================================================
bar_with_color = "white"
bar_wo_color   = "0.65"   # grey fill
edge_color     = "0.1"

# Make two sub-bars inside each wedge by splitting its angular width
subbar_width = slice_width * 0.38   # each bar width
gap          = slice_width * 0.06   # small gap between the two
offset       = (subbar_width + gap) / 2.0  # angular offset from center

def get_range(df, ccsn, b0):
    row = df[(df["CC"] == ccsn) & (df["B_0"] == b0)]
    if row.empty:
        return None
    r = row.iloc[0]["range"]
    if (
        r is None
        or not isinstance(r, (tuple, list))
        or len(r) != 2
        or not np.isfinite(r[0])
        or not np.isfinite(r[1])
    ):
        return None
    rmin, rmax = float(r[0]), float(r[1])
    if rmax <= rmin:
        return None
    return rmin, rmax

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):
        slice_index  = ic * slices_per_sector + ib * slices_per_band
        theta_center = (slice_index + 0.5) * slice_width

        # WITH CCOs goes "left" in the wedge
        r_w = get_range(df_with, ccsn, b0)
        if r_w is not None:
            rmin, rmax = r_w
            ax.bar(
                theta_center - offset,
                rmax - rmin,
                width=subbar_width,
                bottom=rmin,
                color=bar_with_color,
                edgecolor=edge_color,
                linewidth=0.7,
                zorder=3,
            )

        # WITHOUT CCOs goes "right" in the wedge
        r_wo = get_range(df_wo, ccsn, b0)
        if r_wo is not None:
            rmin, rmax = r_wo
            ax.bar(
                theta_center + offset,
                rmax - rmin,
                width=subbar_width,
                bottom=rmin,
                color=bar_wo_color,
                edgecolor=edge_color,
                linewidth=0.7,
                zorder=3,
            )

# ============================================================
# 4. LABEL INITIAL MAGNETIC FIELDS
# ============================================================
B_0_labels = [
    r"$10^{14}\,\mathrm{G}$",
    r"$1.5 \times 10^{14}\,\mathrm{G}$",
    r"$2 \times 10^{14}\,\mathrm{G}$",
    r"$2.5 \times 10^{14}\,\mathrm{G}$",
]
b0_label_map = dict(zip(field_order, B_0_labels))

label_radius = r_max_data + 3

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):
        band_start   = ic * slices_per_sector + ib * slices_per_band
        band_end     = band_start + slices_per_band
        center_slice = (band_start + band_end) / 2.0
        theta_center = center_slice * slice_width

        angle_deg = np.degrees(np.pi/2 - theta_center) + 180
        label = b0_label_map.get(b0, f"{b0:g}")

        ax.text(
            theta_center,
            label_radius,
            label,
            ha="center",
            va="center",
            color="black",
            fontsize=17,
            rotation=angle_deg,
            rotation_mode="anchor",
            zorder=30,
        )

# ============================================================
# 5. CONCENTRIC CIRCLES
# ============================================================
r_ticks = [11, 21, 31, 41, 51, 61, 71, 81, 91]
labels  = [10, 20, 30, 40, 50, 60, 70, 80, 90]
important_labels = {10, 30, 50, 70, 90}

ax.set_yticks(r_ticks)
ax.set_yticklabels([])
ax.grid(color="white", linestyle="--", linewidth=0.4, alpha=0.5)

theta = np.linspace(0, 2*np.pi, 400)

for r, lab in zip(r_ticks, labels):
    if lab in important_labels:
        ax.plot(theta, np.full_like(theta, r), color="white",
                linewidth=1.0, linestyle=":", alpha=0.9, zorder=5)

theta_label = np.deg2rad(45)
for r, lab in zip(r_ticks, labels):
    ax.text(
        theta_label,
        r,
        f"{lab}%",
        ha="center",
        va="center",
        color="white",
        fontsize=13 if lab in important_labels else 8,
        fontweight="bold" if lab in important_labels else "normal",
        zorder=10
    )

# ============================================================
# 6. LEGEND
# ============================================================
legend_handles = [
    Patch.Patch(facecolor=bar_with_color, edgecolor="black",
                label="Magnetar fraction (with CCOs)"),
    Patch.Patch(facecolor=bar_wo_color, edgecolor="black",
                label="Magnetar fraction (without CCOs)"),
]

fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    frameon=False,
    fontsize=13
)

plt.tight_layout()
plt.savefig(f'/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/plot_piechart_magnetar_fraction_without_cco.pdf')

plt.show()


In [ ]:
df

In [ ]:
df.columns

In [ ]:
df_sorted = df.sort_values(
    by=['CC/SN', 'initial_magnetic_field'],
    ascending=[True, True]   # A ascending, B descending
)

In [ ]:
df_sorted

# Counter clockwise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as Patch

# ============================================================
# 1. DATA
# ============================================================
# df = pd.read_csv("your_file.csv")  # if not already defined

# Make sure df is sorted in the order you want (increasing CC/SN, B0)
df = df.sort_values(by=["CC/SN", "initial_magnetic_field"])

# Long format: one row per (B0, CC/SN, category)
def parse_value(v):
    """
    '0'       -> None (no bar)
    '60'      -> (0, 60)
    '[30-60]' -> (30, 60)
    """
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if v == 0:
            return None
        return (0.0, float(v))

    if isinstance(v, str):
        s = v.strip()
        if s == "" or s == "0":
            return None

        if s.startswith("[") and s.endswith("]") and "-" in s:
            lo, hi = s[1:-1].split("-")
            return float(lo), float(hi)

        try:
            val = float(s)
            if val == 0:
                return None
            return (0.0, val)
        except ValueError:
            return None

    return None

long = df.melt(
    id_vars=["initial_magnetic_field", "CC/SN"],
    var_name="category",
    value_name="value",
)
long["range"] = long["value"].map(parse_value)

field_order = list(dict.fromkeys(long["initial_magnetic_field"]))  # order of B0
ccsn_order  = list(dict.fromkeys(long["CC/SN"]))                    # order of CC/SN
cat_order   = [
    "RPP + Magnetars",
    "RPP + Magnetars + XDINS",
    "RPP + Magnetars + XDINS + CCO",
]

n_cc   = len(ccsn_order)   # 3
n_b0   = len(field_order)  # 4
n_cat  = len(cat_order)    # 3

total_slices = n_cc * n_b0 * n_cat   # 36
r_max = 100
slice_width = 2 * np.pi / total_slices

# ============================================================
# 2. FIGURE & BACKGROUND GRADIENT
# ============================================================
base_colors = [
    "#E76F51",   # reddish for CC/SN 1
    "#457B9D",   # blueish for CC/SN 2
    "#2A9D8F",   # greenish for CC/SN 3
]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
ax.spines['polar'].set_visible(False)
ax.spines['polar'].set_linewidth(0)

ax.set_theta_direction(-1)   # clockwise increasing angle
r_plot = 120                 # a bit beyond 100 for long lines
ax.set_ylim(0, r_plot)

ax.set_xticks([])

def make_shades(hex_color):
    """Return 4 shades (brightest → darkest) from a base hex color."""
    r, g, b = to_rgb(hex_color)
    darkest = (r * 0.45, g * 0.45, b * 0.45)
    dark    = (r * 0.65, g * 0.65, b * 0.65)
    medium  = (r * 0.80, g * 0.80, b * 0.80)
    light   = (r * 0.93, g * 0.93, b * 0.93)

    # leftmost band (smallest angle in the sector) = lightest
    # then darker as angle increases clockwise
    return [light, medium, dark, darkest]

slices_per_sector = 12          # per CC/SN
slices_per_band   = 3           # per B0, 3 slices = 3 categories
bands_per_sector  = 4           # 4 B0 values

# draw background gradient (now CC/SN and B0 increase clockwise)
for sector_idx in range(3):
    sector_start = sector_idx * slices_per_sector
    shades = make_shades(base_colors[sector_idx])

    for band_idx in range(bands_per_sector):
        band_start_slice = sector_start + band_idx * slices_per_band
        band_end_slice   = band_start_slice + slices_per_band

        start_theta = band_start_slice * slice_width
        width_theta = (band_end_slice - band_start_slice) * slice_width
        center_theta = start_theta + width_theta / 2.0

        ax.bar(
            center_theta,
            r_max,
            width=width_theta,
            bottom=0,
            color=shades[band_idx],
            edgecolor="none",
            alpha=0.8,
            zorder=0,
        )

# ---- plot the 36 radial lines ----
for k in range(total_slices):
    theta = k * slice_width

    # default line length
    r_end = r_max
    lw = 0.8

    # CC/SN boundaries every 12 slices → longer + thicker
    if k % slices_per_sector == 0:
        r_end = r_plot
        lw = 2.0

    ax.plot(
        [theta, theta],
        [0, r_end],
        color="black",
        linewidth=lw,
        zorder=5
    )

# ============================================================
# 3. OVERLAY GREY BARS FROM DATA (NOW SAME CLOCKWISE ORDER)
# ============================================================
bar_colors = {
    "RPP + Magnetars": "0.2",                  # dark grey
    "RPP + Magnetars + XDINS": "0.5",            # medium grey
    "RPP + Magnetars + XDINS + CCO": "0.9",    # very light grey / white
}

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):
        for it, cat in enumerate(cat_order):

            slice_index = ic * slices_per_sector + ib * slices_per_band + it
            # NO 2*pi -  : increasing CC/SN and B0 clockwise
            theta_center = (slice_index + 0.5) * slice_width

            row = long[
                (long["CC/SN"] == ccsn)
                & (long["initial_magnetic_field"] == b0)
                & (long["category"] == cat)
            ]

            if row.empty:
                continue

            r_range = row.iloc[0]["range"]
            if r_range is None:
                continue

            r_min, r_max_bar = r_range
            height = r_max_bar - r_min

            this_color = bar_colors[cat]

            ax.bar(
                theta_center,
                height,
                width=slice_width * 0.9,
                bottom=r_min,
                color=this_color,
                edgecolor="0.1",
                linewidth=0.6,
                zorder=3,
            )

# ============================================================
# 4. LABEL INITIAL MAGNETIC FIELDS (one label per 3-slice band)
# ============================================================
bfield_labels = {
    5e13: r"$5 \times 10^{13}$ G",
    1e14: r"$10^{14}$ G",
    5e14: r"$5 \times 10^{14}$ G",
    1e15: r"$10^{15}$ G",
}

label_radius = r_max + 3

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):

        band_start   = ic * slices_per_sector + ib * slices_per_band
        band_end     = band_start + slices_per_band
        center_slice = (band_start + band_end) / 2.0  # e.g. 1.5, 4.5, ...

        # center of band, same clockwise order as wedges
        theta_center = center_slice * slice_width

        angle_deg = np.degrees(np.pi/2 - theta_center) + 180  # upside down, as you wanted

        ax.text(
            theta_center,
            label_radius,
            bfield_labels[b0],
            ha="center",
            va="center",
            color="black",
            fontsize=13,
            rotation=angle_deg,
            rotation_mode="anchor",
            zorder=30,
        )

# ============================================================
# 5. CONCENTRIC CIRCLES WITH SUBTLE HIGHLIGHTS AT 10,30,50,70,90
# ============================================================
r_ticks = [11, 21, 31, 41, 51, 61, 71, 81, 91]
labels  = [10, 20, 30, 40, 50, 60, 70, 80, 90]
important_labels = {10, 30, 50, 70, 90}

ax.set_yticks(r_ticks)
ax.set_yticklabels([])
ax.grid(color="white", linestyle="--", linewidth=0.4, alpha=0.5)

theta = np.linspace(0, 2*np.pi, 400)

for r, lab in zip(r_ticks, labels):
    if lab in important_labels:
        ax.plot(
            theta,
            np.full_like(theta, r),
            color="white",
            linewidth=1.0,
            linestyle=":",
            alpha=0.9,
            zorder=5,
        )

theta_label = np.deg2rad(45)

for r, lab in zip(r_ticks, labels):
    ax.text(
        theta_label,
        r,
        str(lab) + '%',
        ha="center",
        va="center",
        color="white",
        fontsize=13 if lab in important_labels else 8,
        fontweight="bold" if lab in important_labels else "normal",
        zorder=10
    )

# ============================================================
# 6. LEGEND
# ============================================================
legend_handles = [
    Patch.Patch(facecolor="0.2", edgecolor="black", label="RPP + Magnetars"),
    Patch.Patch(facecolor="0.5", edgecolor="black", label="RPP + Magnetars + XDINS"),
    Patch.Patch(facecolor="0.9", edgecolor="black", label="RPP + Magnetars + XDINS + CCO"),
]

fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    frameon=False,
    fontsize=13
)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as Patch

# ============================================================
# 1. DATA
# ============================================================
# df = pd.read_csv("your_file.csv")  # if not already defined

# Make sure df is sorted in the order you want (increasing CC/SN, B0)
df = df.sort_values(by=["CC/SN", "initial_magnetic_field"])

# Long format: one row per (B0, CC/SN, category)
def parse_value(v):
    """
    '0'       -> None (no bar)
    '60'      -> (0, 60)
    '[30-60]' -> (30, 60)
    """
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if v == 0:
            return None
        return (0.0, float(v))

    if isinstance(v, str):
        s = v.strip()
        if s == "" or s == "0":
            return None

        if s.startswith("[") and s.endswith("]") and "-" in s:
            lo, hi = s[1:-1].split("-")
            return float(lo), float(hi)

        try:
            val = float(s)
            if val == 0:
                return None
            return (0.0, val)
        except ValueError:
            return None

    return None

long = df.melt(
    id_vars=["initial_magnetic_field", "CC/SN"],
    var_name="category",
    value_name="value",
)
long["range"] = long["value"].map(parse_value)

field_order = list(dict.fromkeys(long["initial_magnetic_field"]))  # order of B0
ccsn_order  = list(dict.fromkeys(long["CC/SN"]))                    # order of CC/SN
cat_order   = [
    "RPP + Magnetars",
    "RPP + Magnetars + XDINS",
    "RPP + Magnetars + XDINS + CCO",
]

n_cc   = len(ccsn_order)   # 3
n_b0   = len(field_order)  # 4
n_cat  = len(cat_order)    # 3

total_slices = n_cc * n_b0 * n_cat   # 36
r_max = 100
slice_width = 2 * np.pi / total_slices

# ============================================================
# 2. FIGURE & BACKGROUND GRADIENT
# ============================================================
base_colors = [
    "#E76F51",   # reddish for CC/SN 1
     "skyblue",   # blueish for CC/SN 2
    "#2A9D8F",   # greenish for CC/SN 3
]



fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
ax.spines['polar'].set_visible(False)
ax.spines['polar'].set_linewidth(0)

ax.set_theta_direction(-1)   # clockwise increasing angle
r_plot = 120                 # a bit beyond 100 for long lines
ax.set_ylim(0, r_plot)

ax.set_xticks([])

def make_shades(hex_color):
    """Return 4 shades (brightest → darkest) from a base hex color."""
    r, g, b = to_rgb(hex_color)
    darkest = (r * 0.45, g * 0.45, b * 0.45)
    dark    = (r * 0.65, g * 0.65, b * 0.65)
    medium  = (r * 0.80, g * 0.80, b * 0.80)
    light   = (r * 0.93, g * 0.93, b * 0.93)

    # leftmost band (smallest angle in the sector) = lightest
    # then darker as angle increases clockwise
    return [light, medium, dark, darkest]

slices_per_sector = 12          # per CC/SN
slices_per_band   = 3           # per B0, 3 slices = 3 categories
bands_per_sector  = 4           # 4 B0 values

###---------------------CC SN colors ------------------------------------------

for sector_idx in range(3):
    sector_start = sector_idx * slices_per_sector
    shades = make_shades(base_colors[sector_idx])
    
    CC_SN_labels = ["CC SN $= 1$","CC SN $= 2$","CC SN $= 3$"]
    angles_text_cc = [np.pi/4, np.pi, np.pi+np.pi/1.5]
    rotation_text = [225,90,-30]
    
    ax.text(
            angles_text_cc[sector_idx],
            r_plot,
            CC_SN_labels[sector_idx],
            ha="center",
            va="center",
            color=shades[3],
            fontsize=16,
            rotation= rotation_text[sector_idx],
            rotation_mode="anchor",
            zorder=30,
        )

    
    for band_idx in range(bands_per_sector):
        band_start_slice = sector_start + band_idx * slices_per_band
        band_end_slice   = band_start_slice + slices_per_band

        start_theta = band_start_slice * slice_width
        width_theta = (band_end_slice - band_start_slice) * slice_width
        center_theta = start_theta + width_theta / 2.0

        ax.bar(
            center_theta,
            r_max,
            width=width_theta,
            bottom=0,
            color=shades[band_idx],
            edgecolor="none",
            alpha=0.8,
            zorder=0,
        )

       
# ---- plot the 36 radial lines ----
for k in range(total_slices):
    theta = k * slice_width

    # default line length
    r_end = r_max
    lw = 0.4

    # CC/SN boundaries every 12 slices → longer + thicker
    if k % slices_per_sector == 0:
        r_end = r_plot
        lw = 2.0
    # B0 (magnetic field) boundaries every 3 slices → medium thickness
    elif k % slices_per_band == 0:
        r_end = r_max      # same length as normal grid lines
        lw = 1.4
        
    ax.plot(
        [theta, theta],
        [0, r_end],
        color="black",
        linewidth=lw,
        zorder=5
    )

# ============================================================
# 3. OVERLAY GREY BARS FROM DATA (NOW SAME CLOCKWISE ORDER)
# ============================================================
bar_colors = {
    "RPP + Magnetars": "0.2",                  # very light grey / white
    "RPP + Magnetars + XDINS": "0.5",            # medium grey
    "RPP + Magnetars + XDINS + CCO": "1",    # dark grey
}

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):
        for it, cat in enumerate(cat_order):

            slice_index = ic * slices_per_sector + ib * slices_per_band + it
            # NO 2*pi -  : increasing CC/SN and B0 clockwise
            theta_center = (slice_index + 0.5) * slice_width

            row = long[
                (long["CC/SN"] == ccsn)
                & (long["initial_magnetic_field"] == b0)
                & (long["category"] == cat)
            ]

            if row.empty:
                continue

            r_range = row.iloc[0]["range"]
            if r_range is None:
                continue

            r_min, r_max_bar = r_range
            height = r_max_bar - r_min

            this_color = bar_colors[cat]

            ax.bar(
                theta_center,
                height,
                width=slice_width * 0.9,
                bottom=r_min,
                color=this_color,
                edgecolor="0.1",
                linewidth=0.6,
                zorder=3,
            )

# ============================================================
# 4. LABEL INITIAL MAGNETIC FIELDS (one label per 3-slice band)
# ============================================================
bfield_labels = {
    5e13: r"$5 \times 10^{13}$ G",
    1e14: r"$10^{14}$ G",
    5e14: r"$5 \times 10^{14}$ G",
    1e15: r"$10^{15}$ G",
}

label_radius = r_max + 3

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):

        band_start   = ic * slices_per_sector + ib * slices_per_band
        band_end     = band_start + slices_per_band
        center_slice = (band_start + band_end) / 2.0  # e.g. 1.5, 4.5, ...

        # center of band, same clockwise order as wedges
        theta_center = center_slice * slice_width

        angle_deg = np.degrees(np.pi/2 - theta_center) + 180  # upside down, as you wanted

        ax.text(
            theta_center,
            label_radius,
            bfield_labels[b0],
            ha="center",
            va="center",
            color="black",
            fontsize=16,
            rotation=angle_deg,
            rotation_mode="anchor",
            zorder=30,
        )

# ============================================================
# 5. CONCENTRIC CIRCLES WITH SUBTLE HIGHLIGHTS AT 10,30,50,70,90
# ============================================================
r_ticks = [11, 21, 31, 41, 51, 61, 71, 81, 91]
labels  = [10, 20, 30, 40, 50, 60, 70, 80, 90]
important_labels = {10, 30, 50, 70, 90}

ax.set_yticks(r_ticks)
ax.set_yticklabels([])
ax.grid(color="white", linestyle=(0, (1, 4)), linewidth=0.7, alpha=1)

theta = np.linspace(0, 2*np.pi, 400)

for r, lab in zip(r_ticks, labels):
    if lab in important_labels:
        ax.plot(
            theta,
            np.full_like(theta, r),
            color="white",
            linewidth=1.1,
            linestyle=":",
            alpha=0.9,
            zorder=5,
        )

#-------- Labels for each concentric circle---------------

theta_label = np.deg2rad(0) # where to put the labels at theta = 0

for r, lab in zip(r_ticks, labels):
    ax.text(
        theta_label,
        r,
        '\n'+'\n'+str(lab) + '%' if lab in important_labels else '\n'+str(lab) + '%',
        ha="center",
        va="center",
        color="white",
        fontsize=14 if lab in important_labels else 10,
        fontweight="bold" if lab in important_labels else "normal",
        zorder=10
    )

# ============================================================
# 6. LEGEND
# ============================================================
legend_handles = [
    Patch.Patch(facecolor="0.2", edgecolor="black", label="RPP + Magnetars"),
    Patch.Patch(facecolor="0.5", edgecolor="black", label="RPP + Magnetars + XDINS"),
    Patch.Patch(facecolor="0.9", edgecolor="black", label="RPP + Magnetars + XDINS + CCO"),
]

fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    frameon=False,
    fontsize=15
)
plt.tight_layout()
plt.savefig(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/pie_chart_magnetar_fraction_results.pdf",
    bbox_inches='tight'
)
plt.show()


In [ ]:

# ============================================================
# 1. DATA
# ============================================================
# If df is already defined, skip this and just keep using your df.
# Example:
# df = pd.read_csv("your_file.csv")

# Make sure df is sorted in the order you want:
df = df.sort_values(by=["CC/SN", "initial_magnetic_field"])

# Long format: one row per (B0, CC/SN, category)
def parse_value(v):
    """
    '0'       -> None (no bar)
    '60'      -> (0, 60)
    '[30-60]' -> (30, 60)
    """
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if v == 0:
            return None
        return (0.0, float(v))

    if isinstance(v, str):
        s = v.strip()
        if s == "" or s == "0":
            return None

        if s.startswith("[") and s.endswith("]") and "-" in s:
            lo, hi = s[1:-1].split("-")
            return float(lo), float(hi)

        try:
            val = float(s)
            if val == 0:
                return None
            return (0.0, val)
        except ValueError:
            return None

    return None


long = df.melt(
    id_vars=["initial_magnetic_field", "CC/SN"],
    var_name="category",
    value_name="value",
)
long["range"] = long["value"].map(parse_value)

field_order = list(dict.fromkeys(long["initial_magnetic_field"]))  # order of B0
ccsn_order  = list(dict.fromkeys(long["CC/SN"]))                    # order of CC/SN
cat_order   = [
    "RPP+Magnetars",
    "RPP+Magnetars + CCO",
    "RPP+Magnetars + CCO + XDINS",
]

n_cc   = len(ccsn_order)   # 3
n_b0   = len(field_order)  # 4
n_cat  = len(cat_order)    # 3

total_slices = n_cc * n_b0 * n_cat   # 36
r_max = 100
slice_width = 2 * np.pi / total_slices

# ============================================================
# 2. FIGURE & BACKGROUND GRADIENT
# ============================================================
base_colors = [
    "#E76F51",   # reddish for CC/SN 1
    "#457B9D",   # blueish for CC/SN 2
    "#2A9D8F",   # greenish for CC/SN 3
]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
ax.spines['polar'].set_visible(False)   # ← hides the outer circle
ax.spines['polar'].set_linewidth(0)

ax.set_theta_direction(-1)
r_max = 100
r_plot = 120          # a bit beyond 100 for the long lines
ax.set_ylim(0, r_plot)

ax.set_xticks([])

def make_shades(hex_color):
    """Return 4 shades (darkest → lightest) from a base hex color."""
    r, g, b = to_rgb(hex_color)
    darkest = (r * 0.45, g * 0.45, b * 0.45)
    dark    = (r * 0.65, g * 0.65, b * 0.65)
    medium  = (r * 0.80, g * 0.80, b * 0.80)
    light   = (r * 0.93, g * 0.93, b * 0.93)
    return [darkest, dark, medium, light]

slices_per_sector = 12          # per CC/SN
slices_per_band   = 3           # per B0 (one shade), 3 slices = 3 categories
bands_per_sector  = 4           # 4 B0 values

# draw background gradient
for sector_idx in range(3):
    sector_start = sector_idx * slices_per_sector
    shades = make_shades(base_colors[sector_idx])

    for band_idx in range(bands_per_sector):
        band_start_slice = sector_start + band_idx * slices_per_band
        band_end_slice   = band_start_slice + slices_per_band

        start_theta = band_start_slice * slice_width
        width_theta = (band_end_slice - band_start_slice) * slice_width
        center_theta = start_theta + width_theta / 2.0

        ax.bar(
            center_theta,
            r_max,
            width=width_theta,
            bottom=0,
            color=shades[band_idx],
            edgecolor="none",
            alpha=0.9,
            zorder=0,
        )

# ---- plot the 36 radial lines ----
for k in range(total_slices):
    theta = k * slice_width

    # default line length
    r_end = r_max
    lw = 0.8

    # CC/SN boundaries every 12 slices → longer + thicker
    if k % slices_per_sector == 0:
        r_end = r_plot
        lw = 2.0

    ax.plot(
        [theta, theta],
        [0, r_end],
        color="black",
        linewidth=lw,
        zorder=5
    )


# ============================================================
# 3. OVERLAY GREY BARS FROM DATA
# ============================================================
# mapping: slice_index = ic*12 + ib*3 + it
# where ic = CC/SN index, ib = B0 index, it = category index
bar_colors = {
    "RPP + Magnetars": "0.2",                  # dark grey
    "RPP + Magnetars + CCO": "0.5",            # medium grey
    "RPP + Magnetars + CCO + XDINS": "0.9",    # very light grey / white
}

for ic, ccsn in enumerate(ccsn_order):
    for ib, b0 in enumerate(field_order):
        for it, cat in enumerate(cat_order):

            slice_index = ic * slices_per_sector + ib * slices_per_band + it
            theta_center = 2*np.pi - (slice_index + 0.5) * slice_width

            row = long[
                (long["CC/SN"] == ccsn)
                & (long["initial_magnetic_field"] == b0)
                & (long["category"] == cat)
            ]

            if row.empty:
                continue

            r_range = row.iloc[0]["range"]
            if r_range is None:
                continue

            r_min, r_max_bar = r_range
            height = r_max_bar - r_min

            # ------ color by category ------
            this_color = bar_colors[cat]

            ax.bar(
                theta_center,
                height,
                width=slice_width * 0.9,
                bottom=r_min,
                color=this_color,
                edgecolor="0.1",
                linewidth=0.6,
                zorder=3,
            )



# ============================================================
# Initial magnetic field labels
# ============================================================
# ============================================================
# 5. LABEL INITIAL MAGNETIC FIELDS (one label per 3-slice band)
# ============================================================

bfield_labels = {
    5e13: r"$5 \times 10^{13}$ G",
    1e14: r"$10^{14}$ G",
    5e14: r"$5 \times 10^{14}$ G",
    1e15: r"$10^{15}$ G",
}

label_radius = r_max + 3    # distance from center (tweak if needed)

for ic, ccsn in enumerate(ccsn_order):              # 0..2
    for ib, b0 in enumerate(field_order):           # 0..3

        # compute center slice of the 3-slice band
        band_start   = ic * slices_per_sector + ib * slices_per_band
        band_end     = band_start + slices_per_band
        center_slice = (band_start + band_end) / 2

        # counter-clockwise angle (your reversed order)
        theta_center = 2*np.pi - (center_slice ) * slice_width

        # rotation angle so text is upright relative to radius
        angle_deg = np.degrees(np.pi/2 - theta_center) + 180


        ax.text(
            theta_center,
            label_radius,
            bfield_labels[b0],    # formatted LaTeX label
            ha="center",
            va="center",
            color="black",
            fontsize=13,
            rotation=angle_deg,
            rotation_mode="anchor",
            zorder=30,
        )

# ============================================================
# CONCENTRIC CIRCLES WITH SUBTLE HIGHLIGHTS AT 10,30,50,70,90
# ============================================================

r_ticks = [11, 21, 31, 41, 51, 61, 71, 81, 91]
labels  = [10, 20, 30, 40, 50, 60, 70, 80, 90]
important_labels = {10, 30, 50, 70, 90}

# 1. baseline faint grid
ax.set_yticks(r_ticks)
ax.set_yticklabels([])
ax.grid(color="white", linestyle="--", linewidth=0.5, alpha=0.4)

# 2. subtle highlighted circles for 10,30,50,70,90
theta = np.linspace(0, 2*np.pi, 400)

for r, lab in zip(r_ticks, labels):
    if lab in important_labels:
        ax.plot(
            theta,
            np.full_like(theta, r),
            color="white",
            linewidth=1.0, 
            linestyle=":",# reduced thickness
            alpha=0.8,        # softer
            zorder=5,
        )

# 3. labels at angle -132°, bold only for important ones
theta_label = np.deg2rad(-132)

for r, lab in zip(r_ticks, labels):
    ax.text(
        theta_label,
        r,
        str(lab)+'%',
        ha="center",
        va="center",
        color="white",
        fontsize=10 if lab in important_labels else 8,
        fontweight="bold" if lab in important_labels else "normal",
        zorder=10
    )



legend_handles = [
    Patch.Patch(facecolor="0.2", edgecolor="black", label="RPP + Magnetars"),
    Patch.Patch(facecolor="0.5", edgecolor="black", label="RPP + Magnetars + CCO"),
    Patch.Patch(facecolor="0.9", edgecolor="black", label="RPP + Magnetars + CCO + XDINS"),
]

fig.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),   # outside the figure, not shrinking the plot
    frameon=False,
    fontsize=13
)
plt.tight_layout()
plt.show()
